# LeetCode #102 — Binary Tree Level Order Traversal: BFS theo từng tầng

**Mức độ:** Medium · **Chủ đề:** Tree, Breadth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def levelOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
```

Mục tiêu của notebook:

1. Hiểu **BFS (Breadth-First Search)** với hàng đợi `collections.deque`.
2. Mẹo **"chụp kích thước tầng"**: `for _ in range(len(q))` để tách các tầng.
3. Cách 2: **DFS mang theo độ sâu** cũng cho kết quả theo tầng.
4. Khuôn mẫu BFS tầng dùng cho rất nhiều bài cây khác.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân, trả về các giá trị theo **thứ tự từng tầng**: từ trên xuống dưới, trong mỗi tầng từ trái sang phải. Mỗi tầng là một list.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 2000]`.
- `-1000 <= Node.val <= 1000`

---

## Ví dụ

```text
    3
   / \
  9  20          ->  [[3], [9, 20], [15, 7]]
    /  \
   15   7

root = [1]  ->  [[1]]
root = []   ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Hàng đợi (queue) và `collections.deque`
3. Duyệt theo chiều rộng (BFS — Breadth-First Search)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Hàng đợi (queue) và `collections.deque`

### Hàng đợi là gì?

Hàng đợi là cấu trúc **vào trước, ra trước** (FIFO — First In, First Out), giống xếp hàng mua vé: người đến trước được phục vụ trước. Thêm ở **cuối**, lấy ra ở **đầu**.

```text
enqueue 1, 2, 3:     đầu → [1, 2, 3] ← cuối
dequeue → 1:         đầu → [2, 3]
enqueue 4:           đầu → [2, 3, 4]
```

### Vì sao không dùng `list.pop(0)`?

`list` lưu liền nhau; bỏ phần tử đầu buộc **dời** mọi phần tử còn lại sang trái → `O(n)`. `collections.deque` (hàng đợi hai đầu) thêm/bỏ ở **cả hai đầu** trong `O(1)`.

| Thao tác | `deque` | `list` |
|---|---|---|
| thêm cuối `append` | `O(1)` | `O(1)` |
| bỏ đầu `popleft` / `pop(0)` | `O(1)` | `O(n)` |
| thêm đầu `appendleft` / `insert(0, x)` | `O(1)` | `O(n)` |

In [6]:
from collections import deque
import time as _time_queue

q = deque()
for x in [1, 2, 3]:
    q.append(x)                  # vào cuối
print("hàng đợi:", q)
print("popleft:", q.popleft(), "->", q)   # ra đầu
q.appendleft(0)
print("appendleft(0):", q, "| xem đầu:", q[0], "| xem cuối:", q[-1], "| len:", len(q))

_n = 50_000
_t0 = _time_queue.perf_counter()
_l = list(range(_n))
while _l:
    _l.pop(0)
_t1 = _time_queue.perf_counter()
_d = deque(range(_n))
while _d:
    _d.popleft()
_t2 = _time_queue.perf_counter()
print(f"bỏ đầu {_n} lần: list.pop(0) {_t1 - _t0:.3f}s | deque.popleft {_t2 - _t1:.3f}s")

hàng đợi: deque([1, 2, 3])
popleft: 1 -> deque([2, 3])
appendleft(0): deque([0, 2, 3]) | xem đầu: 0 | xem cuối: 3 | len: 3


bỏ đầu 50000 lần: list.pop(0) 0.272s | deque.popleft 0.005s


### Khi nào dùng hàng đợi?

- **BFS (duyệt theo chiều rộng)**: xử lý các đỉnh theo thứ tự được phát hiện — gần trước, xa sau.
- Mô phỏng hàng chờ, xử lý tác vụ theo lượt.
- `deque` cũng làm được ngăn xếp (`append` + `pop`) và **cửa sổ trượt** (bỏ phần tử cũ ở đầu).

Lỗi hay gặp: `q.pop()` (bỏ **cuối**) thay vì `q.popleft()` → vô tình biến hàng đợi thành ngăn xếp, BFS thành DFS.

## 2.3 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [7]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [8]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [9]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.4 Áp dụng vào bài này

Đây chính là **BFS theo tầng** trên **cây nhị phân**: **hàng đợi** `deque` chứa các nút của tầng hiện tại; ghi `len(q)`, lấy đúng chừng ấy nút ra, đẩy con của chúng vào — được một tầng.

# 3. Nền tảng

## 3.1 `TreeNode` và dựng cây

In [10]:
from typing import List, Optional
from collections import deque, defaultdict
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

## 3.2 Hàng đợi: vì sao dùng `deque` chứ không phải `list`?

Hàng đợi (queue) là **vào trước ra trước (FIFO)**: thêm ở cuối, lấy ở đầu.

- `list.pop(0)` phải dời mọi phần tử còn lại → `O(n)` mỗi lần.
- `deque.popleft()` → `O(1)`.

In [11]:
import timeit
n = 50_000
t_list = timeit.timeit(lambda: [lst.pop(0) for lst in [list(range(n))] for _ in range(n)], number=1)
t_deque = timeit.timeit(lambda: [dq.popleft() for dq in [deque(range(n))] for _ in range(n)], number=1)
print(f"list.pop(0) x {n}: {t_list:.3f}s | deque.popleft() x {n}: {t_deque:.4f}s")

list.pop(0) x 50000: 0.346s | deque.popleft() x 50000: 0.0072s


# 4. BFS thường (chưa tách tầng)

BFS lấy nút ở đầu hàng đợi, thêm các con của nó vào cuối. Kết quả là thứ tự theo tầng nhưng **gộp chung** một list.

In [12]:
def bfs_flat(root: Optional[TreeNode]) -> List[int]:
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        out.append(node.val)
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    return out

print(bfs_flat(build_tree([3, 9, 20, None, None, 15, 7])))

[3, 9, 20, 15, 7]


Làm sao biết tầng nào kết thúc ở đâu?

# 5. Ý tưởng chính: chụp kích thước tầng

**Bất biến:** ở đầu mỗi vòng `while`, hàng đợi chứa **đúng toàn bộ** các nút của một tầng (và không gì khác).

Vậy `size = len(q)` chính là số nút của tầng hiện tại. Lấy ra đúng `size` nút (các con của chúng được thêm vào cuối, thuộc tầng sau) → sau vòng `for`, hàng đợi chứa đúng tầng tiếp theo.

```text
q = [root]
while q:
    level = []
    for _ in range(len(q)):        # len(q) được tính MỘT LẦN ở đầu vòng for
        node = q.popleft()
        level.append(node.val)
        thêm node.left, node.right (nếu có) vào q
    result.append(level)
```

In [13]:
def level_order_bfs(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(level)
    return result

print(level_order_bfs(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [9, 20], [15, 7]]


`range(len(q))` tính `len(q)` **một lần** khi tạo `range` — việc `q` dài thêm trong lúc lặp không ảnh hưởng số vòng. Đây chính là "chụp" kích thước.

# 6. Mô phỏng

In [14]:
def trace_levels(values: list) -> List[List[int]]:
    root = build_tree(values)
    result, q = [], deque([root]) if root else deque()
    depth = 0
    while q:
        size = len(q)
        print(f"Tầng {depth}: q = {[n.val for n in q]} -> size = {size}")
        level = []
        for _ in range(size):
            node = q.popleft()
            level.append(node.val)
            kids = [c.val for c in (node.left, node.right) if c]
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
            print(f"   lấy {node.val}, thêm con {kids} -> q = {[n.val for n in q]}")
        result.append(level)
        depth += 1
    print("Kết quả:", result)
    return result

_ = trace_levels([3, 9, 20, None, None, 15, 7])

Tầng 0: q = [3] -> size = 1
   lấy 3, thêm con [9, 20] -> q = [9, 20]
Tầng 1: q = [9, 20] -> size = 2
   lấy 9, thêm con [] -> q = [20]
   lấy 20, thêm con [15, 7] -> q = [15, 7]
Tầng 2: q = [15, 7] -> size = 2
   lấy 15, thêm con [] -> q = [7]
   lấy 7, thêm con [] -> q = []
Kết quả: [[3], [9, 20], [15, 7]]


# 7. Cách 2: DFS mang theo độ sâu

Không cần hàng đợi: DFS (pre-order: gốc → trái → phải) và đưa `node.val` vào `result[depth]`. Vì ta luôn thăm **trái trước phải**, trong cùng một tầng các nút được thêm từ trái sang phải.

In [15]:
def level_order_dfs(root: Optional[TreeNode]) -> List[List[int]]:
    result = []
    def dfs(node, depth):
        if not node:
            return
        if depth == len(result):          # lần đầu tới tầng này
            result.append([])
        result[depth].append(node.val)
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return result

print(level_order_dfs(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [9, 20], [15, 7]]


DFS đệ quy có độ sâu bằng **chiều cao** cây — cây lệch 2000 nút sẽ vượt giới hạn đệ quy mặc định (1000) của Python. BFS không có vấn đề này.

# 8. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def levelOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
        if not root:
            return []
        result, q = [], deque([root])
        while q:
            level = []
            for _ in range(len(q)):
                node = q.popleft()
                level.append(node.val)
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
            result.append(level)
        return result

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual if len(str(actual)) < 60 else str(actual)[:57] + '...'}")


def random_tree(n):
    if n == 0:
        return None
    k = random.randint(0, n - 1)
    return TreeNode(random.randint(-9, 9), random_tree(k), random_tree(n - 1 - k))


def brute_levels(root):
    """Gán (độ sâu, vị trí trái-phải) cho mỗi nút rồi gom nhóm."""
    groups = defaultdict(list)
    def walk(node, depth, path):
        if node:
            groups[depth].append((path, node.val))
            walk(node.left, depth + 1, path + "0")
            walk(node.right, depth + 1, path + "1")
    walk(root, 0, "")
    return [[v for _, v in sorted(groups[d])] for d in range(len(groups))]


sol = Solution()
check("Ví dụ 1", sol.levelOrder(build_tree([3, 9, 20, None, None, 15, 7])), [[3], [9, 20], [15, 7]])
check("Ví dụ 2", sol.levelOrder(build_tree([1])), [[1]])
check("Ví dụ 3", sol.levelOrder(build_tree([])), [])
check("Lệch trái", sol.levelOrder(build_tree([1, 2, None, 3])), [[1], [2], [3]])

chain = None
for v in range(2000):
    chain = TreeNode(v, chain)
check("Chuỗi 2000 nút", len(sol.levelOrder(chain)), 2000)

random.seed(102)
for _ in range(3000):
    root = random_tree(random.randint(0, 15))
    expected = brute_levels(root)
    assert sol.levelOrder(root) == expected == level_order_dfs(root)
print("3000 cây ngẫu nhiên: BFS và DFS khớp cách gom nhóm theo (độ sâu, đường đi)")

Ví dụ 1            PASS  [[3], [9, 20], [15, 7]]
Ví dụ 2            PASS  [[1]]
Ví dụ 3            PASS  []
Lệch trái          PASS  [[1], [2], [3]]
Chuỗi 2000 nút     PASS  2000


3000 cây ngẫu nhiên: BFS và DFS khớp cách gom nhóm theo (độ sâu, đường đi)

`brute_levels` sắp xếp các nút cùng độ sâu theo chuỗi đường đi `"0"`/`"1"` (trái = 0, phải = 1) — cùng độ dài nên thứ tự từ điển chính là thứ tự trái sang phải.

# 9. Vì sao đúng? Độ phức tạp?

**Bất biến** (đầu mỗi vòng `while`): `q` chứa đúng các nút ở độ sâu `d`, theo thứ tự trái sang phải.

- Ban đầu: `q = [root]` — tầng 0.
- Vòng `for` lấy ra đúng các nút tầng `d` theo thứ tự, và thêm con của chúng (trái trước phải) → tầng `d + 1` theo đúng thứ tự trái sang phải.

```text
Time:  O(n) — mỗi nút vào và ra hàng đợi đúng một lần
Space: O(w) — w là độ rộng lớn nhất của một tầng (tối đa khoảng n/2 với cây đầy đủ)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Dùng `len(q)` trong điều kiện `while` bên trong

In [18]:
def wrong_dynamic_len(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level, i = [], 0
        while i < len(q):                         # SAI: len(q) tăng trong lúc lặp
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(level)
    return result

print("Sai :", wrong_dynamic_len(build_tree([3, 9, 20, None, None, 15, 7])), "| Đúng:", sol.levelOrder(build_tree([3, 9, 20, None, None, 15, 7])))

Sai : [[3, 9, 20, 15, 7]] | Đúng: [[3], [9, 20], [15, 7]]


`while i < len(q)` với `i` không tăng và `q` liên tục thay đổi → gộp mọi tầng thành một. Phải chụp `size = len(q)` **trước**.

## Lỗi 2: Thêm `None` vào hàng đợi

Nếu `q.append(node.left)` mà không kiểm tra, lát sau `node.val` với `node = None` gây `AttributeError`.

In [19]:
def wrong_append_none(root: Optional[TreeNode]) -> List[List[int]]:
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)                 # lỗi khi node là None
            q.append(node.left)                    # SAI: không kiểm tra None
            q.append(node.right)
        result.append(level)
    return result

try:
    wrong_append_none(build_tree([1, 2]))
except AttributeError as e:
    print("Lỗi:", e)

Lỗi: 'NoneType' object has no attribute 'val'


## Lỗi 3: Không xử lý `root = None`

`deque([None])` → vòng đầu lấy ra `None` → lỗi. Kiểm tra `if not root: return []`.

## Lỗi 4: DFS thăm phải trước trái

Thứ tự trong mỗi tầng sẽ bị đảo ngược.

# 11. Mẹo nhận diện pattern

**BFS theo tầng** khi đề nói "từng tầng", "theo mức", "gần nhất", "ít bước nhất":

```python
q = deque([root])
while q:
    for _ in range(len(q)):
        node = q.popleft()
        ... xử lý ...
        thêm con vào q
    ... kết thúc một tầng ...
```

Bài dùng ngay khuôn mẫu này: **#107 Level Order II** (đảo kết quả), **#103 Zigzag**, **#199 Right Side View** (phần tử cuối mỗi tầng), **#637 Average of Levels**, **#515 Largest Value in Each Row**, **#111 Minimum Depth** (tầng đầu tiên gặp lá), **#116 Populating Next Right Pointers**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Viết level order cho cây `[1, 2, 3, 4, None, None, 5, None, 6]`.

## Bài 2 — Điền chỗ trống

In [20]:
def exercise_2_level_order(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level = []
        # TODO: lặp đúng số nút của tầng hiện tại
        result.append(level)
    return result


# assert exercise_2_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
# assert exercise_2_level_order(build_tree([])) == []

## Bài 3 — Biến thể: Binary Tree Right Side View (LeetCode #199)

Đứng bên phải cây, trả về giá trị các nút nhìn thấy được từ trên xuống (nút **cuối** mỗi tầng).

```python
right_side_view([1, 2, 3, None, 5, None, 4]) -> [1, 3, 4]
right_side_view([1, 2, 3, 4])                -> [1, 3, 4]
```

In [21]:
def exercise_3_right_side_view(root: Optional[TreeNode]) -> List[int]:
    # TODO
    pass


# assert exercise_3_right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
# assert exercise_3_right_side_view(build_tree([1, 2, 3, 4])) == [1, 3, 4]
# assert exercise_3_right_side_view(build_tree([])) == []

## Bài 4 — Biến thể: Minimum Depth of Binary Tree (LeetCode #111)

Số nút trên đường ngắn nhất từ gốc tới **lá** gần nhất. BFS dừng ngay khi gặp lá đầu tiên.

```python
min_depth([3, 9, 20, None, None, 15, 7])        -> 2
min_depth([2, None, 3, None, 4, None, 5, None, 6]) -> 5
```

In [22]:
def exercise_4_min_depth(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_4_min_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 2
# assert exercise_4_min_depth(build_tree([2, None, 3, None, 4, None, 5, None, 6])) == 5
# assert exercise_4_min_depth(build_tree([])) == 0

## Bài 5 — Mini challenge: Maximum Width of Binary Tree (LeetCode #662)

Độ rộng của một tầng = khoảng cách giữa nút trái nhất và phải nhất **tính cả các vị trí trống** ở giữa (như trong cây đầy đủ). Trả về độ rộng lớn nhất.

```python
width_of_tree([1, 3, 2, 5, 3, None, 9])                        -> 4
width_of_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])         -> 7
```

Gợi ý nhanh: đánh số vị trí như heap: gốc `0`, con trái `2i`, con phải `2i + 1`. Độ rộng tầng = `cuối − đầu + 1`.

In [23]:
def exercise_5_width_of_tree(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_5_width_of_tree(build_tree([1, 3, 2, 5, 3, None, 9])) == 4
# assert exercise_5_width_of_tree(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])) == 7
# assert exercise_5_width_of_tree(build_tree([1, 3, 2, 5])) == 2

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_levels` sau khi tự làm.

**Bài 3**
- Mức 1: dùng khung BFS; sau vòng `for`, `level[-1]` là nút phải nhất.

**Bài 4**
- Mức 1: đếm `depth` từ 1; trong vòng `for`, gặp nút **không có con nào** thì trả `depth`.

**Bài 5**
- Mức 1: hàng đợi chứa cặp `(node, idx)`.
- Mức 2: để số không quá lớn, có thể trừ `idx` của nút đầu tầng khỏi mọi chỉ số (Python không tràn số, nhưng trong ngôn ngữ khác thì cần).

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
_ = trace_levels([1, 2, 3, 4, None, None, 5, None, 6])

Tầng 0: q = [1] -> size = 1
   lấy 1, thêm con [2, 3] -> q = [2, 3]
Tầng 1: q = [2, 3] -> size = 2
   lấy 2, thêm con [4] -> q = [3, 4]
   lấy 3, thêm con [5] -> q = [4, 5]
Tầng 2: q = [4, 5] -> size = 2
   lấy 4, thêm con [6] -> q = [5, 6]
   lấy 5, thêm con [] -> q = [6]
Tầng 3: q = [6] -> size = 1
   lấy 6, thêm con [] -> q = []
Kết quả: [[1], [2, 3], [4, 5], [6]]


## Đáp án Bài 2

In [25]:
def answer_2_level_order(root: Optional[TreeNode]) -> List[List[int]]:
    return level_order_bfs(root)

assert answer_2_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert answer_2_level_order(build_tree([])) == []
print("Bài 2 PASS (giống level_order_bfs ở mục 5)")

Bài 2 PASS (giống level_order_bfs ở mục 5)


## Đáp án Bài 3

In [26]:
def answer_3_right_side_view(root: Optional[TreeNode]) -> List[int]:
    return [level[-1] for level in level_order_bfs(root)]

assert answer_3_right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert answer_3_right_side_view(build_tree([1, 2, 3, 4])) == [1, 3, 4]
assert answer_3_right_side_view(build_tree([])) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_min_depth(root: Optional[TreeNode]) -> int:
    if not root:
        return 0
    q, depth = deque([root]), 1
    while q:
        for _ in range(len(q)):
            node = q.popleft()
            if not node.left and not node.right:
                return depth
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        depth += 1

assert answer_4_min_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 2
assert answer_4_min_depth(build_tree([2, None, 3, None, 4, None, 5, None, 6])) == 5
assert answer_4_min_depth(build_tree([])) == 0
print("Bài 4 PASS")

Bài 4 PASS


BFS dừng ngay ở tầng có lá đầu tiên — không cần duyệt hết cây (khác DFS phải đi mọi nhánh).

## Đáp án Bài 5

In [28]:
def answer_5_width_of_tree(root: Optional[TreeNode]) -> int:
    if not root:
        return 0
    best, q = 0, deque([(root, 0)])
    while q:
        first_idx = q[0][1]
        last_idx = q[-1][1]
        best = max(best, last_idx - first_idx + 1)
        for _ in range(len(q)):
            node, idx = q.popleft()
            idx -= first_idx
            if node.left:
                q.append((node.left, 2 * idx))
            if node.right:
                q.append((node.right, 2 * idx + 1))
    return best

assert answer_5_width_of_tree(build_tree([1, 3, 2, 5, 3, None, 9])) == 4
assert answer_5_width_of_tree(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])) == 7
assert answer_5_width_of_tree(build_tree([1, 3, 2, 5])) == 2
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> BFS với `deque`; mỗi vòng ngoài xử lý **đúng một tầng** nhờ `for _ in range(len(q))` (chụp kích thước trước khi thêm con). Hoặc DFS với tham số `depth`, thăm trái trước phải.

---

## Checklist trước khi nộp

- [ ] `root = None` → `[]`?
- [ ] Chụp `len(q)` trước khi lặp tầng?
- [ ] Không thêm `None` vào hàng đợi?
- [ ] Dùng `deque.popleft()`, không `list.pop(0)`?